# Frequency calibration workflow

This notebook focuses on the two frequency-calibration loops you will use
most often after initial bring-up: qubit control frequency and readout
frequency. Adjust the setup cell for your system before you run it.

## 1. Create an `Experiment`

Start with one qubit whose readout and drive channels are already wired
and visible in the configuration.

In [ ]:
import numpy as np

import qubex as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect and collect a baseline

A waveform check plus one Rabi measurement gives you a good baseline
before you change the frequencies.

In [ ]:
exp.connect()

waveform_result = exp.check_waveform()
rabi_result = exp.obtain_rabi_params()

## 3. Run a coarse control-frequency sweep

Use a broad chevron scan first so you can see the control-frequency region before the narrower calibration step.

In [ ]:
# Obtain the Chevron pattern
chevron_result = exp.chevron_pattern(
    exp.qubit_labels,
    detuning_range=np.linspace(-0.05, 0.05, 51),
    time_range=np.arange(0, 201, 4),
)

## 4. Calibrate the control frequency

Sweep a detuning window around the current qubit frequency and let Qubex
fit the updated resonance.

In [ ]:
control_frequency_result = exp.calibrate_control_frequency(
    targets=exp.qubit_labels,
    detuning_range=np.linspace(-0.01, 0.01, 21),
    time_range=np.arange(0, 101, 4),
)

In [ ]:
# Update `control_frequency.yaml` manually, then reload
exp.reload()

## 5. Re-check the Rabi oscillation

Running `obtain_rabi_params()` again quickly shows whether the updated
qubit frequency moved the oscillation closer to the expected model.

In [ ]:
updated_rabi_result = exp.obtain_rabi_params()

## 6. Calibrate the readout frequency

Use a narrower detuning window once the qubit drive side is in place.

In [ ]:
readout_frequency_result = exp.calibrate_readout_frequency(
    targets=exp.qubit_labels,
    detuning_range=np.linspace(-0.01, 0.01, 21),
    time_range=np.arange(0, 101, 4),
)

In [ ]:
# Update `readout_frequency.yaml` manually, then reload
exp.reload()

## 7. Save the calibration note

Save the updated calibration note so later notebooks start from the newly
fitted frequencies.

In [ ]:
exp.calib_note.save()